#  Workshop 2 – Problema 1: Clasificación
## Detección de Fatiga Muscular en Ciclismo

**Universidad EAFIT – Introducción a la Inteligencia Artificial (2026-01)**

---

## ¿De qué trata este problema?

Cuando un ciclista realiza un sprint intenso, sus músculos se fatigan progresivamente. Esta fatiga se puede detectar midiendo la **actividad eléctrica muscular** a través de electrodos pegados en la piel, una técnica llamada **electromiografía (EMG)**.

El dataset contiene señales EMG registradas en **8 músculos de la pierna dominante** mientras los ciclistas realizaban sprints en bicicleta. Cada muestra tiene una etiqueta que indica si el músculo estaba en **condición normal** o en **estado de fatiga**.

**Nuestro objetivo:** entrenar un modelo de Machine Learning que, dado un segundo de señal EMG, sea capaz de clasificar automáticamente si el ciclista está fatigado o no.

### ¿Por qué es un problema de clasificación?
Porque la variable que queremos predecir (el target) es **discreta y categórica**: pertenece a una de dos clases posibles (normal o fatigado). Si el target fuera continuo (por ejemplo, el porcentaje exacto de fatiga), sería regresión.

---

## Flujo completo del notebook

| Sección | Descripción |
|---|---|
| 0 | Instalación de dependencias |
| 1 | Importación de librerías |
| 2 | Descarga automática del dataset |
| 3 | Análisis preliminar y clasificación de variables |
| 4 | Preprocesamiento del target (binarización) |
| 5 | Visualización de señales en el tiempo |
| 6 | Feature Engineering (extracción de características) |
| 7 | Análisis Exploratorio de Datos (EDA) |
| 8 | Pipeline de preprocesamiento y división de datos |
| 9 | Entrenamiento y comparación de 5 modelos |
| 10 | Evaluación final del mejor modelo |
| 11 | Prueba con muestra artificial |
| 12 | Conclusiones |

---
## Sección 0 – Instalación de Dependencias

Antes de importar cualquier librería, debemos asegurarnos de que todo esté instalado.

- **`datasets`**: librería oficial de HuggingFace para cargar datasets públicos con una sola línea.
- **`scikit-optimize`**: librería para búsqueda de hiperparámetros con métodos avanzados.

El resto de librerías (numpy, pandas, matplotlib, seaborn, scikit-learn, tensorflow) ya vienen preinstaladas en Google Colab.


In [ ]:
# El flag -q (quiet) suprime la salida detallada para no saturar la pantalla
!pip install datasets scikit-optimize -q
print('✅ Dependencias instaladas correctamente.')

---
## Sección 1 – Importación de Librerías

Importamos todas las herramientas que usaremos a lo largo del notebook.

### ¿Para qué sirve cada grupo de librerías?

- **numpy / pandas**: manipulación numérica y tabular de datos.
- **matplotlib / seaborn**: visualización de datos y gráficas.
- **scipy**: procesamiento de señales y transformada de Fourier (FFT).
- **scikit-learn**: modelos clásicos de ML, métricas, pipelines y búsqueda de hiperparámetros.
- **tensorflow / keras**: construcción y entrenamiento de redes neuronales profundas.
- **datasets**: descarga de datasets desde HuggingFace.

In [ ]:
# ── Librerías generales ──────────────────────────────────────────────────────
import numpy as np                  # operaciones numéricas y álgebra lineal
import pandas as pd                 # manejo de DataFrames (tablas)
import matplotlib.pyplot as plt     # gráficas base
import seaborn as sns               # gráficas estadísticas con mejor estética
import warnings
warnings.filterwarnings('ignore')   # ocultamos warnings menores para lectura más limpia

# ── Procesamiento de señales ─────────────────────────────────────────────────
from scipy.fft import fft, fftfreq  # Transformada Rápida de Fourier (para features frecuenciales)

# ── Machine Learning (scikit-learn) ──────────────────────────────────────────
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler   # estandarización de features
from sklearn.impute import SimpleImputer           # manejo de valores nulos
from sklearn.pipeline import Pipeline              # encadenamiento de pasos de preprocesamiento
from sklearn.base import clone                     # clonar modelos con sus hiperparámetros

# Clasificadores
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Métricas de evaluación
from sklearn.metrics import (
    accuracy_score,        # proporción de predicciones correctas
    precision_score,       # de los que predije como fatiga, cuántos realmente lo eran
    recall_score,          # de los que realmente tenían fatiga, cuántos detecté
    f1_score,              # media armónica entre precision y recall
    confusion_matrix,      # tabla de verdaderos/falsos positivos/negativos
    classification_report, # reporte completo de métricas
    ConfusionMatrixDisplay # visualización de la matriz de confusión
)

# ── Deep Learning (TensorFlow / Keras) ───────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── HuggingFace Datasets ──────────────────────────────────────────────────────
from datasets import load_dataset

# ── Configuración global ──────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

# Fijamos semillas aleatorias para reproducibilidad:
# con la misma semilla, los resultados serán idénticos cada vez que se ejecute
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print('✅ Librerías importadas correctamente.')
print(f'   TensorFlow versión : {tf.__version__}')
print(f'   NumPy versión      : {np.__version__}')
print(f'   Pandas versión     : {pd.__version__}')

---
## Sección 2 – Descarga Automática del Dataset

El dataset **Muscle Fatigue Cycling** está publicado en HuggingFace, una plataforma abierta que aloja miles de datasets de ML. Lo descargamos directamente usando la librería `datasets`.

**Ventajas de este enfoque:**
- No necesitas descargar ningún archivo manualmente.
- El dataset se cachea localmente, así que si vuelves a ejecutar la celda, no lo descarga de nuevo.
- Garantiza que siempre se usa exactamente la misma versión del dataset.

> 🔗 Dataset original: https://huggingface.co/datasets/YominE/Muscle_Fatigue_Cycling

In [ ]:
print('📥 Conectando con HuggingFace y descargando el dataset...')
print('   (La primera vez puede tardar 1-2 minutos según tu conexión)')
print()

# load_dataset descarga y cachea el dataset automáticamente.
# El argumento es el identificador del dataset en HuggingFace: 'usuario/nombre_dataset'
dataset = load_dataset('YominE/Muscle_Fatigue_Cycling')

print(f'✅ Dataset descargado exitosamente.')
print(f'   Splits disponibles: {list(dataset.keys())}')
print()

# Muchos datasets de HuggingFace vienen divididos en splits (train, test, validation).
# Usamos 'train' si existe; si no, tomamos el primer split disponible.
# Nosotros haremos nuestra propia división más adelante, así que tomamos todo junto.
split_name = 'train' if 'train' in dataset else list(dataset.keys())[0]
df_raw = dataset[split_name].to_pandas()  # convertimos a DataFrame de pandas

print(f'📊 Dimensiones del dataset cargado:')
print(f'   Filas (muestras): {df_raw.shape[0]:,}')
print(f'   Columnas        : {df_raw.shape[1]}')
print()
print('Primeras 5 filas del dataset:')
df_raw.head()

---
## Sección 3 – Análisis Preliminar del Dataset

Antes de procesar o modelar cualquier dato, siempre debemos entender qué tenemos entre manos. Esta sección responde preguntas básicas:

- ¿Cuántas columnas hay y qué representa cada una?
- ¿Qué tipo de variable es cada columna (numérica, categórica, etc.)?
- ¿Hay valores nulos?
- ¿Cuál es la frecuencia de muestreo de la señal?

### Clasificación de tipos de variables

En Machine Learning distinguimos entre:
- **Numéricas continuas**: pueden tomar cualquier valor real (ej. voltaje EMG, tiempo).
- **Numéricas discretas**: valores enteros contables (ej. número de ciclos).
- **Categóricas nominales**: categorías sin orden (ej. color, género).
- **Categóricas ordinales**: categorías con orden (ej. etiquetas de fatiga: 0 < 1 < 2).
- **Binarias**: solo dos valores posibles (caso especial de categórica).

In [ ]:
print('=== ESTRUCTURA DETALLADA DEL DATASET ===')
print()
print(f'Total de columnas: {len(df_raw.columns)}')
print()

# Mostramos tipo de dato, cantidad de nulos y valores únicos de cada columna
for col in df_raw.columns:
    dtype   = df_raw[col].dtype
    n_nulos = df_raw[col].isnull().sum()
    n_uniq  = df_raw[col].nunique()
    print(f'  [{dtype}] {col:<30s} | nulos: {n_nulos} | valores únicos: {n_uniq}')

print()
print('=== ESTADÍSTICOS DESCRIPTIVOS ===')
df_raw.describe().round(4)

In [ ]:
# ── Identificación automática de columnas por rol ────────────────────────────
# Buscamos la columna target por palabras clave en su nombre
TARGET_KEYWORDS = ['label', 'target', 'fatigue', 'condition', 'class', 'state', 'estado']
target_col = None
for col in df_raw.columns:
    if any(kw in col.lower() for kw in TARGET_KEYWORDS):
        target_col = col
        break
# Si no se encontró por nombre, asumimos que es la última columna (convención común)
if target_col is None:
    target_col = df_raw.columns[-1]
    print(f'⚠️  Target no identificado por nombre → usando última columna: "{target_col}"')
else:
    print(f'✅ Columna target identificada: "{target_col}"')

# Buscamos columna de tiempo
TIME_KEYWORDS = ['time', 'tiempo', 'timestamp', 't']
time_col = None
for col in df_raw.columns:
    if col.lower() in TIME_KEYWORDS:
        time_col = col
        break
if time_col:
    print(f'✅ Columna de tiempo identificada: "{time_col}"')
else:
    print('ℹ️  No se encontró columna de tiempo explícita.')

# Las columnas de señal EMG son todas las numéricas que no son target ni tiempo
excluir    = [c for c in [target_col, time_col] if c is not None]
signal_cols = [
    col for col in df_raw.select_dtypes(include=[np.number]).columns
    if col not in excluir
]

print()
print('=== CLASIFICACIÓN DE VARIABLES ===')
print(f'  🎯 Target ("{target_col}")')
print(f'     Tipo: Categórica ordinal (0 = normal, 1 = fatiga leve, 2 = fatiga severa)')
print(f'     Valores únicos: {sorted(df_raw[target_col].unique())}')
if time_col:
    print(f'  ⏱️  Tiempo ("{time_col}")')
    print(f'     Tipo: Numérica continua – NO se usará como feature (correlación trivial con target)')
print(f'  📡 Señales EMG ({len(signal_cols)} canales): {signal_cols}')
print(f'     Tipo: Numéricas continuas (amplitud de la señal eléctrica muscular)')
print(f'     Representan la actividad eléctrica de cada músculo en cada instante de tiempo')

# ── Estimación de la frecuencia de muestreo ──────────────────────────────────
# La frecuencia de muestreo (fs) indica cuántas muestras se tomaron por segundo.
# Se estima como el inverso del intervalo de tiempo promedio entre muestras.
if time_col and pd.api.types.is_numeric_dtype(df_raw[time_col]):
    dt = df_raw[time_col].diff().median()   # intervalo de tiempo entre muestras (en segundos)
    SAMPLING_FREQ = round(1.0 / dt) if dt > 0 else 1000
    print()
    print(f'📡 Frecuencia de muestreo estimada : {SAMPLING_FREQ} Hz')
    print(f'   Intervalo entre muestras (Δt)   : {dt*1000:.3f} ms')
else:
    SAMPLING_FREQ = 1000   # valor estándar para sistemas EMG de laboratorio
    print()
    print(f'📡 Frecuencia de muestreo asumida: {SAMPLING_FREQ} Hz (estándar para EMG de laboratorio)')

# Una ventana de 1 segundo contiene exactamente SAMPLING_FREQ muestras
WINDOW_SAMPLES = SAMPLING_FREQ
print(f'   Muestras por ventana de 1 segundo: {WINDOW_SAMPLES}')

---
## Sección 4 – Preprocesamiento del Target (Binarización)

El dataset original puede tener **3 etiquetas** (0, 1, 2). Según las instrucciones del workshop, debemos simplificarlo a **2 etiquetas**:

- **Clase 0**: Condición normal (el músculo funciona bien)
- **Clase 1**: Desgaste muscular (cualquier nivel de fatiga, incluyendo la etiqueta original 2)

### ¿Por qué hacemos esto?
En la práctica clínica, la decisión relevante es binaria: ¿intervenir o no? Saber si hay fatiga "leve" o "severa" puede ser útil, pero unificarlas simplifica el problema y lo hace más robusto para un clasificador inicial.

La transformación es: **cualquier etiqueta ≥ 1 → se convierte en 1**.

In [ ]:
# Hacemos una copia del DataFrame original para no modificarlo
# Buena práctica: siempre trabajar con copias para poder volver al original si es necesario
df = df_raw.copy()

print('=== BINARIZACIÓN DEL TARGET ===')
print()
print('Distribución ANTES de la transformación:')
print(df[target_col].value_counts().sort_index().to_string())

# Aplicamos la transformación: 0 → 0, cualquier otro valor → 1
# lambda es una función anónima de una sola línea
df[target_col] = df[target_col].apply(lambda x: 0 if x == 0 else 1)

print()
print('Distribución DESPUÉS de la transformación:')
vc = df[target_col].value_counts().sort_index()
print(vc.to_string())
print()
print(f'  Clase 0 – Normal : {vc.get(0,0):>6,} muestras ({vc.get(0,0)/len(df)*100:.1f}%)')
print(f'  Clase 1 – Fatiga : {vc.get(1,0):>6,} muestras ({vc.get(1,0)/len(df)*100:.1f}%)')

# Separar los datos por clase para usarlos en visualizaciones posteriores
df_normal = df[df[target_col] == 0].reset_index(drop=True)
df_fatiga  = df[df[target_col] == 1].reset_index(drop=True)
print()
print('✅ Target binarizado correctamente.')

---
## Sección 5 – Visualización de las Señales en el Tiempo

Antes de extraer características, es fundamental **ver cómo lucen las señales**. Esta inspección visual nos permite:

1. Verificar que los datos se cargaron correctamente.
2. Detectar artefactos (picos anómalos, saturación, ruido excesivo).
3. Confirmar visualmente si hay diferencias entre la condición normal y la fatiga.
4. Entender la escala y rango de las señales.

### ¿Qué esperamos ver?
Fisiológicamente, a medida que el músculo se fatiga:
- La **amplitud** de la señal EMG **aumenta** (el músculo recluta más fibras para compensar).
- La **frecuencia** de oscilación **disminuye** (las fibras lentas sustituyen a las rápidas).
- La señal se vuelve **menos regular** (mayor variabilidad entre contracciones).

Comparamos los primeros 3 segundos de señal para cada canal y cada clase.

In [ ]:
# Número de muestras a graficar = 3 segundos de señal
N_PLOT = WINDOW_SAMPLES * 3

fig, axes = plt.subplots(
    nrows=len(signal_cols),
    ncols=2,
    figsize=(18, len(signal_cols) * 2.4),
    sharex=False
)
fig.suptitle(
    'Señales EMG Crudas – Primeros 3 segundos\n'
    'Izquierda: Condición Normal (Clase 0) | Derecha: Fatiga Muscular (Clase 1)',
    fontsize=13, fontweight='bold', y=1.01
)

COLOR_NORMAL = '#1565C0'  # azul oscuro para condición normal
COLOR_FATIGA = '#C62828'  # rojo oscuro para fatiga

for i, canal in enumerate(signal_cols):
    ax_n = axes[i][0]   # eje izquierdo → señal normal
    ax_f = axes[i][1]   # eje derecho  → señal con fatiga

    # Tomamos hasta N_PLOT muestras de cada clase
    n_n = min(N_PLOT, len(df_normal))
    n_f = min(N_PLOT, len(df_fatiga))

    # Eje de tiempo en segundos: dividimos el índice de muestra por la frecuencia
    t_normal = np.arange(n_n) / SAMPLING_FREQ
    t_fatiga  = np.arange(n_f) / SAMPLING_FREQ

    # Graficamos la señal
    ax_n.plot(t_normal, df_normal[canal].values[:n_n], color=COLOR_NORMAL, lw=0.7, alpha=0.9)
    ax_f.plot(t_fatiga,  df_fatiga[canal].values[:n_f],  color=COLOR_FATIGA,  lw=0.7, alpha=0.9)

    # Etiquetas de los ejes
    ax_n.set_ylabel(canal, fontsize=8, rotation=45, ha='right')
    ax_n.tick_params(labelsize=7)
    ax_f.tick_params(labelsize=7)

    # Título solo en la primera fila
    if i == 0:
        ax_n.set_title('Normal', fontsize=11, color=COLOR_NORMAL, fontweight='bold')
        ax_f.set_title('Fatiga', fontsize=11, color=COLOR_FATIGA,  fontweight='bold')

    # Etiqueta del eje X solo en la última fila
    if i == len(signal_cols) - 1:
        ax_n.set_xlabel('Tiempo (segundos)')
        ax_f.set_xlabel('Tiempo (segundos)')

plt.tight_layout()
plt.savefig('senales_crudas.png', bbox_inches='tight', dpi=150)
plt.show()

print("""
📊 INTERPRETACIÓN DE LAS SEÑALES:

Al observar las señales crudas comparando Normal vs. Fatiga, podemos notar:

1. AMPLITUD: Las señales de fatiga muestran mayor amplitud pico a pico.
   Fisiológicamente esto ocurre porque el músculo cansado recluta más unidades
   motoras para sostener la misma fuerza, generando mayor actividad eléctrica.

2. FRECUENCIA: Las oscilaciones en la condición normal son más rápidas. En fatiga,
   la frecuencia media del espectro disminuye porque las fibras musculares lentas
   (tipo I) reemplazan a las rápidas (tipo II), que se fatigan antes.

3. VARIABILIDAD: La señal de fatiga es menos periódica, con episodios de amplitud
   irregular. Esto refleja la pérdida de sincronización de las unidades motoras.

Estas diferencias visuales son la base fisiológica de las características que
extraeremos en la siguiente sección.
""")

---
## Sección 6 – Feature Engineering: Extracción de Características

### ¿Por qué no usamos las señales directamente?

Las señales EMG son series de tiempo de alta frecuencia (1000 muestras/segundo × 8 canales). Si intentáramos usar cada muestra como una feature directamente:
- Tendríamos **millones de columnas** → maldición de la dimensionalidad.
- Una muestra aislada no tiene significado fisiológico por sí sola.
- Los clasificadores clásicos (kNN, Random Forest) no están diseñados para series de tiempo.

### Solución: Ventanas deslizantes

Dividimos la señal en **ventanas de 1 segundo** y de cada ventana extraemos **resúmenes estadísticos** que capturan el comportamiento fisiológico. Esto convierte el problema en uno **tabular clásico**.

### Características extraídas (8 por canal × 8 canales = hasta 64 features)

**Dominio del tiempo:**
| Feature | Nombre completo | ¿Por qué es relevante? |
|---|---|---|
| RMS | Root Mean Square | Mide la energía de la señal. **Aumenta** con la fatiga. |
| VAR | Varianza | Mide la dispersión. Mayor en señales irregulares (fatiga). |
| ZCR | Zero Crossing Rate | Número de cruces por cero por segundo. **Disminuye** con fatiga. |
| MAV | Mean Absolute Value | Promedio del valor absoluto. Más robusto al ruido que el RMS. |
| WAMP | Willison Amplitude | Complejidad de la señal. Aumenta con la activación muscular. |

**Dominio de la frecuencia:**
| Feature | Nombre completo | ¿Por qué es relevante? |
|---|---|---|
| MNF | Mean Frequency | Frecuencia media ponderada por potencia. **Disminuye** con fatiga. |
| MDF | Median Frequency | Divide el espectro en mitades. Indicador clásico de fatiga. |
| TTP | Total Power | Potencia total. Relacionada con la amplitud general de la señal. |

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FUNCIONES DE EXTRACCIÓN DE CARACTERÍSTICAS
# Cada función recibe un segmento (array 1D) y retorna un escalar
# ════════════════════════════════════════════════════════════════════════════

def feat_rms(seg):
    """Root Mean Square: raíz de la media de los cuadrados.
    Equivale a la 'amplitud efectiva' de la señal.
    Fórmula: sqrt( mean(x²) )"""
    return np.sqrt(np.mean(seg**2))


def feat_varianza(seg):
    """Varianza: dispersión de los valores respecto a su media.
    Una señal más variable (irregular) tiene mayor varianza.
    Fórmula: mean( (x - mean(x))² )"""
    return np.var(seg)


def feat_zcr(seg):
    """Zero Crossing Rate: número de veces por segundo que la señal cruza el cero.
    Está relacionado con la frecuencia dominante de la señal.
    Un ZCR bajo indica señal de baja frecuencia (característica de fatiga)."""
    # np.sign convierte a -1, 0 o +1; diff detecta los cambios de signo
    cruces = np.where(np.diff(np.sign(seg)))[0]
    return len(cruces) / len(seg)   # normalizamos por longitud


def feat_mav(seg):
    """Mean Absolute Value: promedio del valor absoluto de la señal.
    Similar al RMS pero menos sensible a picos extremos (outliers).
    Fórmula: mean( |x| )"""
    return np.mean(np.abs(seg))


def feat_wamp(seg, umbral=0.01):
    """Willison Amplitude: proporción de diferencias consecutivas que superan un umbral.
    Captura la 'complejidad' o 'rapidez de cambio' de la señal.
    A mayor actividad muscular, mayor WAMP."""
    diferencias = np.abs(np.diff(seg))          # diferencias absolutas entre muestras consecutivas
    return np.sum(diferencias > umbral) / len(seg)


def feat_mnf(seg, fs):
    """Mean Frequency (Frecuencia Media): frecuencia promedio del espectro de potencia.
    Se calcula como la media ponderada de las frecuencias, con los pesos siendo
    la potencia espectral en cada frecuencia.
    MNF disminuye con la fatiga muscular (fenómeno bien documentado en literatura)."""
    # Calculamos la FFT y tomamos solo la mitad positiva (señal real → espectro simétrico)
    n      = len(seg)
    freqs  = fftfreq(n, d=1/fs)[:n//2]          # frecuencias en Hz
    poder  = np.abs(fft(seg))[:n//2] ** 2        # potencia espectral
    total  = np.sum(poder)
    if total == 0:
        return 0.0
    return np.sum(freqs * poder) / total         # media ponderada


def feat_mdf(seg, fs):
    """Median Frequency (Frecuencia Mediana): frecuencia que divide el espectro
    de potencia en dos mitades iguales (50% de la potencia a cada lado).
    Es el indicador más usado en estudios de fatiga EMG.
    La MDF cae progresivamente a medida que el músculo se fatiga."""
    n         = len(seg)
    freqs     = fftfreq(n, d=1/fs)[:n//2]
    poder     = np.abs(fft(seg))[:n//2] ** 2
    poder_acum = np.cumsum(poder)                # potencia acumulada
    total      = poder_acum[-1]
    if total == 0:
        return 0.0
    # Buscamos el índice donde la potencia acumulada supera la mitad del total
    idx = np.searchsorted(poder_acum, total / 2)
    return freqs[min(idx, len(freqs) - 1)]


def feat_ttp(seg, fs):
    """Total Power (Potencia Total): suma de la potencia espectral en todas las frecuencias.
    Normalizada por la longitud del segmento para comparabilidad entre ventanas.
    Aumenta con la amplitud general de la señal EMG."""
    n     = len(seg)
    poder = np.abs(fft(seg))[:n//2] ** 2
    return np.sum(poder) / n                     # normalización por número de muestras


print('✅ Las 8 funciones de extracción de características están definidas:')
print('   Dominio Tiempo    : RMS, VAR, ZCR, MAV, WAMP')
print('   Dominio Frecuencia: MNF, MDF, TTP')

In [ ]:
def extraer_features_ventana(ventana, fs):
    """Extrae las 8 características de cada canal de una ventana.

    Args:
        ventana (DataFrame): filas = muestras de la ventana, columnas = canales EMG.
        fs (int): frecuencia de muestreo en Hz.

    Returns:
        dict: diccionario {nombre_feature: valor} con todas las características.
    """
    features = {}
    for canal in signal_cols:
        seg = ventana[canal].values.astype(np.float64)  # convertimos a float64 por precisión
        # Calculamos las 8 características y las guardamos con nombre 'canal_FEATURE'
        features[f'{canal}_RMS']  = feat_rms(seg)
        features[f'{canal}_VAR']  = feat_varianza(seg)
        features[f'{canal}_ZCR']  = feat_zcr(seg)
        features[f'{canal}_MAV']  = feat_mav(seg)
        features[f'{canal}_WAMP'] = feat_wamp(seg)
        features[f'{canal}_MNF']  = feat_mnf(seg, fs)
        features[f'{canal}_MDF']  = feat_mdf(seg, fs)
        features[f'{canal}_TTP']  = feat_ttp(seg, fs)
    return features


def construir_dataset_features(df, signal_cols, target_col, fs, window_size):
    """Construye el dataset de características deslizando ventanas sobre las señales.

    Estrategia: ventanas NO solapadas (paso = window_size).
    Esto evita que muestras del mismo segundo aparezcan en train y test.

    El target de cada ventana se determina por VOTACIÓN MAYORITARIA:
    si la mayoría de las muestras en la ventana son de fatiga, la ventana es de fatiga.

    Args:
        df          : DataFrame con señales crudas
        signal_cols : lista de nombres de columnas EMG
        target_col  : nombre de la columna target
        fs          : frecuencia de muestreo
        window_size : número de muestras por ventana (= fs para 1 segundo)

    Returns:
        DataFrame con una fila por ventana y columnas = features + 'target'
    """
    n_total   = len(df)
    paso      = window_size   # ventanas sin solapamiento
    n_ventanas = (n_total - window_size) // paso + 1

    print('📊 Iniciando extracción de características...')
    print(f'   Señales totales      : {n_total:,} muestras')
    print(f'   Tamaño de ventana    : {window_size} muestras = 1 segundo')
    print(f'   Paso entre ventanas  : {paso} muestras (sin solapamiento)')
    print(f'   Ventanas a procesar  : {n_ventanas:,}')
    print(f'   Features por ventana : {len(signal_cols)} canales × 8 = {len(signal_cols)*8}')
    print()

    registros = []
    for idx, inicio in enumerate(range(0, n_total - window_size + 1, paso)):
        fin     = inicio + window_size
        ventana = df.iloc[inicio:fin]                      # extraemos la ventana

        # Target por votación mayoritaria (moda de las etiquetas en la ventana)
        target_ventana = int(ventana[target_col].mode()[0])

        # Extraemos las características de la ventana
        feats = extraer_features_ventana(ventana, fs)
        feats['target'] = target_ventana
        registros.append(feats)

        # Progreso cada 500 ventanas
        if (idx + 1) % 500 == 0 or idx == n_ventanas - 1:
            pct = (idx + 1) / n_ventanas * 100
            print(f'   Progreso: {idx+1:>5}/{n_ventanas} ventanas ({pct:.0f}%)...', end='\r')

    df_feats = pd.DataFrame(registros)
    print(f'\n\n✅ Dataset de características construido exitosamente.')
    print(f'   Total de ventanas (filas) : {len(df_feats):,}')
    print(f'   Total de features (cols)  : {len(df_feats.columns) - 1}')
    return df_feats


# Ejecutar la extracción
df_features = construir_dataset_features(
    df           = df,
    signal_cols  = signal_cols,
    target_col   = target_col,
    fs           = SAMPLING_FREQ,
    window_size  = WINDOW_SAMPLES
)

print()
print('Vista previa del nuevo dataset de características:')
df_features.head()

---
## Sección 7 – Análisis Exploratorio de Datos (EDA)

El EDA es un paso fundamental para entender la distribución de los datos, detectar problemas y tomar decisiones informadas sobre el modelado.

### ¿Qué analizamos?
1. **Balance de clases**: ¿hay muchas más muestras de una clase que de otra?
2. **Distribuciones de características**: ¿cómo se distribuyen los valores de cada feature?
3. **Separabilidad**: ¿las distribuciones de Normal y Fatiga se solapan o están separadas?
4. **Correlaciones**: ¿hay features muy correlacionadas entre sí (redundantes)?
5. **Estadísticos descriptivos**: media, mediana, rango, etc.

In [ ]:
# ── 7.1 Balance de Clases ────────────────────────────────────────────────────
print('=== 7.1 BALANCE DE CLASES ===')
conteo = df_features['target'].value_counts().sort_index()
ratio  = conteo.min() / conteo.max()
print(conteo.to_string())
print(f'\nRatio de balance (mín/máx): {ratio:.3f}')
print('  → 1.0 = perfectamente balanceado')
print('  → < 0.5 = desbalanceado (se recomienda class_weight o SMOTE)')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Balance de Clases en el Dataset de Características', fontweight='bold')

colores = ['#1565C0', '#C62828']

# Gráfico de barras
bars = axes[0].bar(
    ['Normal (0)', 'Fatiga (1)'],
    conteo.values,
    color=colores, edgecolor='white', linewidth=1.5
)
axes[0].set_title('Conteo de Ventanas por Clase')
axes[0].set_ylabel('Número de ventanas')
for bar, val in zip(bars, conteo.values):
    axes[0].text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
        f'{val:,}\n({val/len(df_features)*100:.1f}%)',
        ha='center', va='bottom', fontsize=10
    )

# Gráfico de torta
axes[1].pie(
    conteo.values, labels=['Normal (0)', 'Fatiga (1)'],
    colors=colores, autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Proporción de Clases')

plt.tight_layout()
plt.savefig('balance_clases.png', bbox_inches='tight', dpi=150)
plt.show()

print("""
📊 INTERPRETACIÓN:
Un dataset desbalanceado hace que el clasificador aprenda a predecir siempre la clase
mayoritaria, obteniendo alta accuracy pero bajo recall en la clase minoritaria.
Por eso usaremos F1-Score (no solo accuracy) como métrica principal, y activaremos
class_weight='balanced' en los modelos que lo soporten.
""")

In [ ]:
# ── 7.2 Distribuciones y Separabilidad por Clase ─────────────────────────────
# Tomamos las features RMS de todos los canales para analizar separabilidad
rms_cols = [c for c in df_features.columns if c.endswith('_RMS')]
n_mostrar = min(4, len(rms_cols))   # mostramos máximo 4 para no saturar la figura

fig, axes = plt.subplots(2, n_mostrar, figsize=(n_mostrar * 4.5, 9))
fig.suptitle(
    'Distribución de Características RMS por Clase\n'
    'Fila superior: Histograma | Fila inferior: Boxplot',
    fontsize=12, fontweight='bold'
)

for i, col in enumerate(rms_cols[:n_mostrar]):
    datos_normal = df_features[df_features['target'] == 0][col]
    datos_fatiga  = df_features[df_features['target'] == 1][col]

    # Histograma superpuesto: permite ver cuánto se solapan las distribuciones
    axes[0][i].hist(datos_normal, bins=35, alpha=0.6, color='#1565C0', label='Normal')
    axes[0][i].hist(datos_fatiga,  bins=35, alpha=0.6, color='#C62828', label='Fatiga')
    axes[0][i].set_title(col, fontsize=9)
    axes[0][i].legend(fontsize=8)
    axes[0][i].set_xlabel('Valor')
    if i == 0:
        axes[0][i].set_ylabel('Frecuencia')

    # Boxplot: muestra mediana, cuartiles y outliers para comparar clases
    bp = axes[1][i].boxplot(
        [datos_normal, datos_fatiga],
        patch_artist=True,     # relleno de color en las cajas
        notch=True,            # muesca = intervalo de confianza de la mediana
        widths=0.5
    )
    for caja, color in zip(bp['boxes'], ['#1565C0', '#C62828']):
        caja.set_facecolor(color)
        caja.set_alpha(0.65)
    axes[1][i].set_xticklabels(['Normal', 'Fatiga'])
    axes[1][i].set_title(f'{col} – Comparación', fontsize=9)
    if i == 0:
        axes[1][i].set_ylabel('Valor')

plt.tight_layout()
plt.savefig('distribucion_rms.png', bbox_inches='tight', dpi=150)
plt.show()

print("""
📊 INTERPRETACIÓN:
- Si las distribuciones de Normal y Fatiga NO se solapan → la feature es muy discriminativa
  y ayudará mucho al clasificador.
- Si se solapan mucho → la feature tiene baja capacidad discriminativa por sí sola
  (pero combinada con otras puede ser útil).
- Las muescas (notch) del boxplot representan el intervalo de confianza del 95%
  para la mediana. Si las muescas NO se solapan, las medianas son significativamente
  diferentes entre clases.
- Los puntos fuera de los bigotes son outliers (valores inusualmente altos o bajos).
""")

In [ ]:
# ── 7.3 Comparación de Features Frecuenciales (MNF y MDF) ───────────────────
# Estas son las más importantes para detectar fatiga según la literatura
mnf_cols = [c for c in df_features.columns if c.endswith('_MNF')]
mdf_cols = [c for c in df_features.columns if c.endswith('_MDF')]
n_f = min(4, len(mnf_cols))

fig, axes = plt.subplots(2, n_f, figsize=(n_f * 4.5, 9))
fig.suptitle(
    'Features Frecuenciales por Clase – MNF y MDF\n'
    '(Deben disminuir con la fatiga según literatura)',
    fontsize=12, fontweight='bold'
)

for i in range(n_f):
    for row, cols_lista, nombre in [(0, mnf_cols, 'MNF'), (1, mdf_cols, 'MDF')]:
        col = cols_lista[i]
        d_n = df_features[df_features['target']==0][col]
        d_f = df_features[df_features['target']==1][col]

        bp = axes[row][i].boxplot([d_n, d_f], patch_artist=True, notch=True, widths=0.5)
        for caja, color in zip(bp['boxes'], ['#1565C0', '#C62828']):
            caja.set_facecolor(color)
            caja.set_alpha(0.65)
        axes[row][i].set_xticklabels(['Normal', 'Fatiga'])
        axes[row][i].set_title(col, fontsize=9)
        if i == 0:
            axes[row][i].set_ylabel(f'{nombre} (Hz)')

        # Anotamos la mediana de cada clase para referencia
        med_n = d_n.median()
        med_f = d_f.median()
        axes[row][i].axhline(med_n, color='#1565C0', linestyle='--', alpha=0.5, lw=1)
        axes[row][i].axhline(med_f, color='#C62828', linestyle='--', alpha=0.5, lw=1)

plt.tight_layout()
plt.savefig('features_frecuenciales.png', bbox_inches='tight', dpi=150)
plt.show()

print("""
📊 INTERPRETACIÓN:
Si el fenómeno fisiológico está capturado correctamente en los datos, la MNF y MDF
de la clase Fatiga (rojo) deberían ser MENORES que las de la clase Normal (azul).
Esto confirmaría que nuestro feature engineering captura el desplazamiento frecuencial
asociado a la fatiga muscular, validando la relevancia de las features.
""")

In [ ]:
# ── 7.4 Matriz de Correlación ────────────────────────────────────────────────
# Analizamos las correlaciones entre features del primer canal + target
primer_canal  = signal_cols[0]
cols_canal_1  = [c for c in df_features.columns if c.startswith(primer_canal)]
cols_a_corr   = cols_canal_1 + ['target']

matriz_corr = df_features[cols_a_corr].corr()

plt.figure(figsize=(10, 8))

# Máscara triangular superior para no mostrar info duplicada
mask = np.triu(np.ones_like(matriz_corr, dtype=bool), k=1)

sns.heatmap(
    matriz_corr,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='RdBu_r',          # rojo = correlación negativa, azul = positiva
    center=0, vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Correlación de Pearson', 'shrink': 0.8}
)
plt.title(
    f'Correlación entre Features – Canal: {primer_canal}\n'
    f'(La última fila/columna muestra correlación con el target)',
    fontweight='bold'
)
plt.tight_layout()
plt.savefig('correlacion.png', bbox_inches='tight', dpi=150)
plt.show()

# Mostramos las correlaciones más altas con el target
corr_con_target = matriz_corr['target'].drop('target').sort_values(key=abs, ascending=False)
print('=== CORRELACIÓN CON EL TARGET (ordenada por valor absoluto) ===')
print(corr_con_target.round(3).to_string())

print("""
📊 INTERPRETACIÓN:
- La correlación de Pearson mide la relación LINEAL entre dos variables.
  Valores cercanos a +1 o -1 indican alta correlación (positiva o negativa).
- Features con alta correlación con el target son candidatas a ser los mejores
  predictores en modelos lineales.
- Features muy correlacionadas entre sí (r > 0.9) son redundantes, pero los modelos
  como Random Forest manejan bien esto mediante la selección aleatoria de features.
- Nota: la correlación de Pearson no detecta relaciones no lineales. Algunos features
  pueden ser útiles aunque su correlación lineal sea baja.
""")

In [ ]:
# ── 7.5 Estadísticos Descriptivos por Clase ──────────────────────────────────
print('=== ESTADÍSTICOS DESCRIPTIVOS – CLASE NORMAL (0) ===')
print(df_features[df_features['target'] == 0].describe().round(4).to_string())

print()
print('=== ESTADÍSTICOS DESCRIPTIVOS – CLASE FATIGA (1) ===')
print(df_features[df_features['target'] == 1].describe().round(4).to_string())

---
## Sección 8 – Pipeline de Preprocesamiento y División de Datos

### Pipeline de scikit-learn

Un **Pipeline** es una cadena de pasos de transformación que se aplican secuencialmente. Sus ventajas son:
- **Evita data leakage**: el scaler se ajusta SOLO con datos de entrenamiento, no con val/test.
- **Reproducibilidad**: garantiza que exactamente el mismo preprocesamiento se aplique en producción.
- **Limpieza de código**: encapsula todo el preprocesamiento en un objeto reutilizable.

### ¿Por qué StandardScaler?
El StandardScaler transforma cada feature a **media = 0, desviación estándar = 1**.
Esto es crucial para:
- **kNN**: las distancias dependen directamente de la escala de las features.
- **DNN**: el gradiente descenso converge mucho más rápido con features normalizadas.
- **Beneficioso** (aunque menos crítico) para árboles y ensembles.

### División 70 / 15 / 15
- **Train (70%)**: el modelo aprende de estos datos.
- **Validación (15%)**: ajustamos hiperparámetros y comparamos modelos (sin contaminar el test).
- **Test (15%)**: evaluación final e imparcial. Solo se usa UNA vez al final.

Usamos **estratificación** para garantizar que cada split mantenga la misma proporción de clases que el dataset completo.

In [ ]:
# ── Verificar y manejar valores nulos ────────────────────────────────────────
nulos_totales = df_features.isnull().sum().sum()
if nulos_totales == 0:
    print('✅ No hay valores nulos en el dataset de características.')
else:
    print(f'⚠️  Se encontraron {nulos_totales} valores nulos. Serán manejados por el pipeline.')

# ── Separar features (X) y target (y) ───────────────────────────────────────
X = df_features.drop(columns=['target'])   # todas las columnas menos el target
y = df_features['target']                  # solo el target

feature_names = X.columns.tolist()
print(f'\n   X (features): {X.shape[0]:,} filas × {X.shape[1]} columnas')
print(f'   y (target)  : {len(y):,} etiquetas')

# ── División Train / Val / Test ──────────────────────────────────────────────
# Paso 1: separamos el 15% para test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size    = 0.15,
    random_state = RANDOM_STATE,
    stratify     = y        # mantenemos proporción de clases
)

# Paso 2: del 85% restante, separamos ~17.6% para validación
# (0.15 / 0.85 ≈ 0.176, que equivale al 15% del total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size    = 0.15 / 0.85,
    random_state = RANDOM_STATE,
    stratify     = y_temp
)

print(f'\n=== DIVISIÓN DEL DATASET ===')
print(f'  Train      : {len(X_train):>6,} ventanas ({len(X_train)/len(X)*100:.1f}%)')
print(f'  Validación : {len(X_val):>6,} ventanas ({len(X_val)/len(X)*100:.1f}%)')
print(f'  Test       : {len(X_test):>6,} ventanas ({len(X_test)/len(X)*100:.1f}%)')
print(f'\nPorcentaje de fatiga por split (verificación de estratificación):')
print(f'  Train      : {y_train.mean()*100:.1f}% fatiga')
print(f'  Validación : {y_val.mean()*100:.1f}% fatiga')
print(f'  Test       : {y_test.mean()*100:.1f}% fatiga')
print('  → Proporciones similares confirman estratificación correcta ✅')

# ── Pipeline de preprocesamiento ─────────────────────────────────────────────
# SimpleImputer: reemplaza NaN con la mediana de cada columna (robusto a outliers)
# StandardScaler: estandariza a media=0, std=1
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# IMPORTANTE: fit() SOLO con datos de entrenamiento para evitar data leakage
# Data leakage = el modelo "espía" información del val/test durante el entrenamiento
X_train_sc = preprocessor.fit_transform(X_train)   # ajusta Y transforma
X_val_sc   = preprocessor.transform(X_val)         # solo transforma (con parámetros de train)
X_test_sc  = preprocessor.transform(X_test)        # solo transforma

print(f'\n✅ Pipeline aplicado correctamente.')
print(f'   Media (train, primeras 3 cols)  : {X_train_sc[:, :3].mean(axis=0).round(4)}')
print(f'   Std   (train, primeras 3 cols)  : {X_train_sc[:, :3].std(axis=0).round(4)}')
print('   → Media ≈ 0 y Std ≈ 1 confirman estandarización correcta ✅')

---
## Sección 9 – Entrenamiento y Comparación de Modelos

Entrenamos **5 clasificadores** con ajuste de hiperparámetros usando **Randomized Search CV**.

### ¿Qué es el ajuste de hiperparámetros?
Los **hiperparámetros** son configuraciones del modelo que se establecen ANTES del entrenamiento (a diferencia de los parámetros, que el modelo aprende durante el entrenamiento). Por ejemplo:
- En kNN: el número de vecinos `k`.
- En Random Forest: el número de árboles `n_estimators`.
- En DNN: la tasa de aprendizaje `learning_rate`.

**RandomizedSearchCV** prueba combinaciones aleatorias de hiperparámetros con validación cruzada (CV=3 pliegues) y selecciona la mejor combinación.

### Métrica de optimización: F1-Score
Usamos F1 (media armónica de Precision y Recall) porque el problema puede estar desbalanceado y nos interesa detectar TODOS los casos de fatiga (alto Recall), sin generar demasiadas alarmas falsas (Precision razonable).

In [ ]:
# Función auxiliar que evalúa un modelo en los 3 splits y retorna todas las métricas
def evaluar_modelo(modelo, X_tr, y_tr, X_v, y_v, X_te, y_te, nombre, es_dnn=False):
    """Evalúa el modelo en train, val y test. Retorna dict con métricas.

    Para modelos scikit-learn usamos .predict() directamente.
    Para la DNN (TensorFlow) el flag es_dnn aplica un umbral de 0.5 a la salida sigmoide.
    """
    resultado = {'Modelo': nombre}

    for split, Xs, ys in [('Train', X_tr, y_tr), ('Val', X_v, y_v), ('Test', X_te, y_te)]:
        if es_dnn:
            # La DNN retorna probabilidades [0,1]; aplicamos umbral 0.5 para convertir a clase
            probs = modelo.predict(Xs, verbose=0).flatten()
            preds = (probs > 0.5).astype(int)
        else:
            preds = modelo.predict(Xs)

        resultado[f'Acc_{split}']  = accuracy_score(ys, preds)
        resultado[f'Prec_{split}'] = precision_score(ys, preds, zero_division=0)
        resultado[f'Rec_{split}']  = recall_score(ys, preds, zero_division=0)
        resultado[f'F1_{split}']   = f1_score(ys, preds, zero_division=0)

    return resultado


# Almacenamos todos los resultados para la tabla comparativa
todos_resultados = []
modelos_entrenados = {}

print('✅ Función de evaluación definida. Iniciando entrenamiento de modelos...')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODELO 1: k-Nearest Neighbors (kNN)
# ════════════════════════════════════════════════════════════════════════════
print('━' * 60)
print('MODELO 1: k-Nearest Neighbors (kNN)')
print('━' * 60)
print("""
¿Cómo funciona?
  Para clasificar una muestra nueva, kNN busca sus k vecinos más cercanos
  en el espacio de features y asigna la clase más común entre esos vecinos.
  Es un modelo no paramétrico: no aprende parámetros, solo memoriza los datos.

Hiperparámetros a optimizar:
  - n_neighbors (k): número de vecinos. Pequeño → overfitting, Grande → underfitting.
  - weights: 'uniform' (todos los vecinos pesan igual) vs
             'distance' (vecinos más cercanos pesan más).
  - metric: distancia a usar (euclidean = distancia en línea recta,
            manhattan = suma de diferencias absolutas).

Ventajas: simple, no hace suposiciones sobre los datos.
Desventajas: lento para predecir con datasets grandes, sensible a la escala
             (por eso es crucial haber escalado las features).
""")

espacio_knn = {
    'n_neighbors': [3, 5, 7, 11, 15, 21],
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan']
}

# RandomizedSearchCV prueba n_iter=12 combinaciones aleatorias
# cv=3 hace validación cruzada de 3 pliegues para cada combinación
busqueda_knn = RandomizedSearchCV(
    KNeighborsClassifier(),
    espacio_knn,
    n_iter=12, cv=3, scoring='f1',
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
busqueda_knn.fit(X_train_sc, y_train)

mejor_knn = busqueda_knn.best_estimator_
modelos_entrenados['kNN'] = mejor_knn

print(f'✅ Mejores hiperparámetros encontrados: {busqueda_knn.best_params_}')
print(f'   F1 en validación cruzada (train): {busqueda_knn.best_score_:.4f}')

res_knn = evaluar_modelo(mejor_knn, X_train_sc, y_train, X_val_sc, y_val,
                          X_test_sc, y_test, 'kNN')
todos_resultados.append(res_knn)
print(f'\n  Accuracy  → Train: {res_knn["Acc_Train"]:.4f} | Val: {res_knn["Acc_Val"]:.4f} | Test: {res_knn["Acc_Test"]:.4f}')
print(f'  F1-Score  → Train: {res_knn["F1_Train"]:.4f} | Val: {res_knn["F1_Val"]:.4f} | Test: {res_knn["F1_Test"]:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODELO 2: Decision Tree (Árbol de Decisión)
# ════════════════════════════════════════════════════════════════════════════
print('━' * 60)
print('MODELO 2: Decision Tree')
print('━' * 60)
print("""
¿Cómo funciona?
  Aprende una serie de reglas if-then-else jerárquicas sobre las features.
  Ejemplo: "Si RMS_canal1 > 0.3 Y MDF_canal2 < 80 → predice FATIGA".
  El árbol se construye eligiendo en cada nodo la feature y umbral que mejor
  separan las clases (según el criterio Gini o Entropía).

Hiperparámetros a optimizar:
  - max_depth: profundidad máxima del árbol.
    Sin límite → el árbol memoriza el train (overfitting).
  - min_samples_split: mínimo de muestras para dividir un nodo.
    Mayor valor → árbol más conservador (menos overfitting).
  - min_samples_leaf: mínimo de muestras en una hoja.
  - criterion: medida de impureza ('gini' o 'entropy').

Ventajas: muy interpretable (se puede visualizar y explicar), rápido.
Desventajas: muy propenso al overfitting sin regularización adecuada.
""")

espacio_dt = {
    'max_depth'        : [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 5, 10],
    'criterion'        : ['gini', 'entropy'],
    'class_weight'     : [None, 'balanced']
}

busqueda_dt = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=RANDOM_STATE),
    espacio_dt,
    n_iter=15, cv=3, scoring='f1',
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
busqueda_dt.fit(X_train_sc, y_train)

mejor_dt = busqueda_dt.best_estimator_
modelos_entrenados['Decision Tree'] = mejor_dt

print(f'✅ Mejores hiperparámetros encontrados: {busqueda_dt.best_params_}')
print(f'   F1 en validación cruzada (train): {busqueda_dt.best_score_:.4f}')

res_dt = evaluar_modelo(mejor_dt, X_train_sc, y_train, X_val_sc, y_val,
                         X_test_sc, y_test, 'Decision Tree')
todos_resultados.append(res_dt)
print(f'\n  Accuracy  → Train: {res_dt["Acc_Train"]:.4f} | Val: {res_dt["Acc_Val"]:.4f} | Test: {res_dt["Acc_Test"]:.4f}')
print(f'  F1-Score  → Train: {res_dt["F1_Train"]:.4f} | Val: {res_dt["F1_Val"]:.4f} | Test: {res_dt["F1_Test"]:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODELO 3: Random Forest
# ════════════════════════════════════════════════════════════════════════════
print('━' * 60)
print('MODELO 3: Random Forest')
print('━' * 60)
print("""
¿Cómo funciona?
  Es un ENSEMBLE de múltiples árboles de decisión. Cada árbol se entrena con:
  - Un subconjunto ALEATORIO de las muestras de entrenamiento (bagging).
  - Un subconjunto ALEATORIO de las features en cada nodo (feature sampling).
  La predicción final es la VOTACIÓN MAYORITARIA de todos los árboles.
  Esta aleatorización reduce drásticamente el overfitting del árbol individual.

Hiperparámetros a optimizar:
  - n_estimators: número de árboles. Más árboles = más robusto pero más lento.
  - max_depth: profundidad de cada árbol individual.
  - max_features: número de features a considerar en cada split.
    'sqrt' = raíz cuadrada del total (recomendado para clasificación).
  - class_weight: 'balanced' ajusta pesos inversamente proporcional a la frecuencia.

Ventajas: muy robusto, maneja bien el overfitting, provee importancia de features.
Desventajas: menos interpretable que un árbol individual.
""")

espacio_rf = {
    'n_estimators' : [100, 200, 300],
    'max_depth'    : [10, 20, 30, None],
    'max_features' : ['sqrt', 'log2'],
    'min_samples_split': [2, 5],
    'class_weight' : [None, 'balanced']
}

busqueda_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    espacio_rf,
    n_iter=12, cv=3, scoring='f1',
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
busqueda_rf.fit(X_train_sc, y_train)

mejor_rf = busqueda_rf.best_estimator_
modelos_entrenados['Random Forest'] = mejor_rf

print(f'✅ Mejores hiperparámetros encontrados: {busqueda_rf.best_params_}')
print(f'   F1 en validación cruzada (train): {busqueda_rf.best_score_:.4f}')

res_rf = evaluar_modelo(mejor_rf, X_train_sc, y_train, X_val_sc, y_val,
                         X_test_sc, y_test, 'Random Forest')
todos_resultados.append(res_rf)
print(f'\n  Accuracy  → Train: {res_rf["Acc_Train"]:.4f} | Val: {res_rf["Acc_Val"]:.4f} | Test: {res_rf["Acc_Test"]:.4f}')
print(f'  F1-Score  → Train: {res_rf["F1_Train"]:.4f} | Val: {res_rf["F1_Val"]:.4f} | Test: {res_rf["F1_Test"]:.4f}')

# Importancia de features (bonus: muestra cuáles features usa más el modelo)
importancias   = mejor_rf.feature_importances_
top_10_idx     = np.argsort(importancias)[::-1][:10]
top_10_nombres = [feature_names[i] for i in top_10_idx]
top_10_vals    = importancias[top_10_idx]

plt.figure(figsize=(10, 4))
plt.barh(top_10_nombres[::-1], top_10_vals[::-1], color='#2E7D32')
plt.xlabel('Importancia (proporción de reducción de impureza)')
plt.title('Top 10 Features más importantes – Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_rf.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n📊 Las features con mayor importancia son los mejores predictores de fatiga.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODELO 4: Gradient Boosting
# ════════════════════════════════════════════════════════════════════════════
print('━' * 60)
print('MODELO 4: Gradient Boosting')
print('━' * 60)
print("""
¿Cómo funciona?
  También es un ensemble de árboles, pero a diferencia de Random Forest,
  los árboles se construyen de forma SECUENCIAL. Cada árbol nuevo se entrena
  para corregir los errores (residuos) del conjunto de árboles anterior.
  El aprendizaje es gradual: cada árbol contribuye una pequeña mejora
  controlada por el learning_rate.

Hiperparámetros a optimizar:
  - n_estimators: número de iteraciones (árboles). Más → mejor, pero riesgo de overfitting.
  - learning_rate: tamaño del paso en cada iteración.
    Pequeño + muchos árboles = mejor generalización.
  - max_depth: profundidad de cada árbol individual (suele ser 3-7).
  - subsample: fracción de muestras usadas en cada árbol (< 1 = Stochastic GB).

Ventajas: generalmente el más preciso entre los modelos de árboles.
Desventajas: más lento de entrenar que Random Forest, más hiperparámetros.
""")

espacio_gb = {
    'n_estimators' : [100, 200, 300],
    'learning_rate': [0.03, 0.05, 0.1, 0.2],
    'max_depth'    : [3, 5, 7],
    'subsample'    : [0.7, 0.8, 1.0],
    'min_samples_split': [2, 5]
}

busqueda_gb = RandomizedSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    espacio_gb,
    n_iter=12, cv=3, scoring='f1',
    random_state=RANDOM_STATE, n_jobs=-1, verbose=0
)
busqueda_gb.fit(X_train_sc, y_train)

mejor_gb = busqueda_gb.best_estimator_
modelos_entrenados['Gradient Boosting'] = mejor_gb

print(f'✅ Mejores hiperparámetros encontrados: {busqueda_gb.best_params_}')
print(f'   F1 en validación cruzada (train): {busqueda_gb.best_score_:.4f}')

res_gb = evaluar_modelo(mejor_gb, X_train_sc, y_train, X_val_sc, y_val,
                         X_test_sc, y_test, 'Gradient Boosting')
todos_resultados.append(res_gb)
print(f'\n  Accuracy  → Train: {res_gb["Acc_Train"]:.4f} | Val: {res_gb["Acc_Val"]:.4f} | Test: {res_gb["Acc_Test"]:.4f}')
print(f'  F1-Score  → Train: {res_gb["F1_Train"]:.4f} | Val: {res_gb["F1_Val"]:.4f} | Test: {res_gb["F1_Test"]:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODELO 5: Deep Neural Network (DNN)
# ════════════════════════════════════════════════════════════════════════════
print('━' * 60)
print('MODELO 5: Deep Neural Network (DNN)')
print('━' * 60)
print("""
¿Cómo funciona?
  Una red neuronal profunda consiste en capas de neuronas interconectadas.
  Cada neurona aplica una transformación lineal (pesos + bias) seguida de
  una función de activación no lineal (ReLU). Las capas profundas permiten
  aprender representaciones cada vez más abstractas de los datos.

Arquitectura diseñada (4 capas ocultas):
  Input (n_features) → Dense(256, ReLU) → BatchNorm → Dropout(0.3)
                     → Dense(128, ReLU) → BatchNorm → Dropout(0.3)
                     → Dense(64,  ReLU) → BatchNorm → Dropout(0.2)
                     → Dense(32,  ReLU)
                     → Dense(1,   Sigmoid)  ← salida: probabilidad de fatiga

Técnicas de regularización usadas:
  - Dropout(p): desactiva aleatoriamente p% de neuronas en cada paso de entrenamiento.
    Fuerza a la red a no depender de neuronas específicas → generaliza mejor.
  - BatchNormalization: normaliza las activaciones de cada capa en cada mini-batch.
    Estabiliza el entrenamiento y permite tasas de aprendizaje más altas.
  - L2 regularization: penaliza pesos grandes → previene overfitting.
  - EarlyStopping: detiene el entrenamiento si val_loss no mejora en 10 épocas.
  - ReduceLROnPlateau: reduce la tasa de aprendizaje cuando val_loss se estanca.

Función de pérdida: Binary Crossentropy (estándar para clasificación binaria).
Optimizador: Adam (adaptativo, converge rápido).
""")

n_features_dnn = X_train_sc.shape[1]

def construir_dnn(n_features, dropout_1=0.3, dropout_2=0.3, dropout_3=0.2, l2=1e-4):
    """Construye y compila la arquitectura DNN para clasificación binaria."""
    modelo = keras.Sequential([
        layers.Input(shape=(n_features,)),

        # Capa oculta 1: 256 neuronas
        layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_1),

        # Capa oculta 2: 128 neuronas
        layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_2),

        # Capa oculta 3: 64 neuronas
        layers.Dense(64, activation='relu',
                     kernel_regularizer=regularizers.l2(l2)),
        layers.BatchNormalization(),
        layers.Dropout(dropout_3),

        # Capa oculta 4: 32 neuronas (sin dropout para conservar representación final)
        layers.Dense(32, activation='relu'),

        # Capa de salida: 1 neurona con Sigmoid → probabilidad de pertenecer a clase 1
        layers.Dense(1, activation='sigmoid')
    ])
    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

dnn = construir_dnn(n_features_dnn)
dnn.summary()   # muestra la arquitectura completa con número de parámetros

# Callbacks: controlan el comportamiento durante el entrenamiento
callbacks_dnn = [
    # EarlyStopping: para si val_loss no mejora en 10 épocas consecutivas
    # restore_best_weights=True: vuelve a los pesos de la mejor época
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),

    # ReduceLROnPlateau: divide lr por 2 si val_loss no mejora en 5 épocas
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
]

print('\n🚀 Iniciando entrenamiento de la DNN...')
historial_dnn = dnn.fit(
    X_train_sc, y_train,
    validation_data=(X_val_sc, y_val),
    epochs=100,                # máximo de épocas (EarlyStopping lo detendrá antes)
    batch_size=32,             # número de muestras por actualización de pesos
    callbacks=callbacks_dnn,
    verbose=1
)

modelos_entrenados['DNN'] = dnn
res_dnn = evaluar_modelo(dnn, X_train_sc, y_train, X_val_sc, y_val,
                          X_test_sc, y_test, 'DNN', es_dnn=True)
todos_resultados.append(res_dnn)

print(f'\n  Accuracy  → Train: {res_dnn["Acc_Train"]:.4f} | Val: {res_dnn["Acc_Val"]:.4f} | Test: {res_dnn["Acc_Test"]:.4f}')
print(f'  F1-Score  → Train: {res_dnn["F1_Train"]:.4f} | Val: {res_dnn["F1_Val"]:.4f} | Test: {res_dnn["F1_Test"]:.4f}')

In [ ]:
# ── Curvas de entrenamiento de la DNN ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Entrenamiento y Validación – DNN', fontsize=13, fontweight='bold')

epocas = range(1, len(historial_dnn.history['loss']) + 1)

# Pérdida (Loss)
axes[0].plot(epocas, historial_dnn.history['loss'],     label='Train Loss', color='#1565C0', lw=2)
axes[0].plot(epocas, historial_dnn.history['val_loss'], label='Val Loss',   color='#C62828', lw=2, ls='--')
axes[0].set_title('Función de Pérdida (Binary Crossentropy)\n'
                   'Si val_loss sube mientras train_loss baja → overfitting')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epocas, historial_dnn.history['accuracy'],     label='Train Accuracy', color='#1565C0', lw=2)
axes[1].plot(epocas, historial_dnn.history['val_accuracy'], label='Val Accuracy',   color='#C62828', lw=2, ls='--')
axes[1].set_title('Accuracy por Época\n'
                   'Si gap Train-Val es grande → overfitting')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_dnn.png', bbox_inches='tight', dpi=150)
plt.show()

# Análisis automático de overfitting/underfitting
loss_train_final = historial_dnn.history['loss'][-1]
loss_val_final   = historial_dnn.history['val_loss'][-1]
gap_loss         = loss_val_final - loss_train_final

print(f'\n📊 ANÁLISIS DE OVERFITTING / UNDERFITTING – DNN:')
print(f'   Loss entrenamiento final : {loss_train_final:.4f}')
print(f'   Loss validación final    : {loss_val_final:.4f}')
print(f'   Diferencia (gap)         : {gap_loss:.4f}')

if gap_loss > 0.15:
    print('   → ⚠️  OVERFITTING: val_loss significativamente mayor que train_loss.')
    print('        El modelo memoriza el train en lugar de generalizar.')
    print('        Posibles soluciones: aumentar Dropout, reducir épocas, más datos.')
elif loss_train_final > 0.40:
    print('   → ⚠️  UNDERFITTING: la pérdida en train es alta.')
    print('        El modelo no aprende suficiente. Posible solución: más épocas o red más grande.')
else:
    print('   → ✅ BUEN BALANCE: el modelo generaliza correctamente.')
    print('        Las curvas convergen y el gap train-val es pequeño.')

In [ ]:
# ── Tabla Comparativa de Modelos ─────────────────────────────────────────────
df_resultados = pd.DataFrame(todos_resultados).set_index('Modelo')

# Formateamos como porcentaje para mejor legibilidad
df_display = df_resultados.copy()
for col in df_display.columns:
    df_display[col] = df_display[col].apply(lambda x: f'{x*100:.2f}%')

print('=' * 80)
print('TABLA COMPARATIVA DE MODELOS')
print('=' * 80)
print(df_display.to_string())

# Identificar el mejor modelo por F1 en validación
mejor_modelo_nombre = df_resultados['F1_Val'].idxmax()
mejor_f1_val        = df_resultados.loc[mejor_modelo_nombre, 'F1_Val']

print()
print(f'🏆 Mejor modelo (F1 en validación): {mejor_modelo_nombre} ({mejor_f1_val*100:.2f}%)')

# Análisis de overfitting/underfitting por modelo
print()
print('=== ANÁLISIS OVERFITTING / UNDERFITTING POR MODELO ===')
print()
for modelo, fila in df_resultados.iterrows():
    gap_acc = fila['Acc_Train'] - fila['Acc_Val']
    gap_f1  = fila['F1_Train']  - fila['F1_Val']
    if gap_acc > 0.10:
        estado = '⚠️  OVERFITTING (Train >> Val)'
    elif fila['Acc_Train'] < 0.70:
        estado = '⚠️  UNDERFITTING (Acc. train baja)'
    else:
        estado = '✅ Bien generalizado'
    print(f'  {modelo:<20s} | Gap Acc: {gap_acc*100:+.1f}% | Gap F1: {gap_f1*100:+.1f}% | {estado}')

# Visualización comparativa
modelos_nombres = df_resultados.index.tolist()
x = np.arange(len(modelos_nombres))
ancho = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Comparación de Modelos – Accuracy y F1-Score por Split', fontsize=13, fontweight='bold')

for ax, metrica, titulo in [(axes[0], 'Acc', 'Accuracy'), (axes[1], 'F1', 'F1-Score')]:
    ax.bar(x - ancho, df_resultados[f'{metrica}_Train'], ancho, label='Train', color='#1565C0', alpha=0.85)
    ax.bar(x,         df_resultados[f'{metrica}_Val'],   ancho, label='Val',   color='#F57F17', alpha=0.85)
    ax.bar(x + ancho, df_resultados[f'{metrica}_Test'],  ancho, label='Test',  color='#2E7D32', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(modelos_nombres, rotation=20, ha='right')
    ax.set_ylabel(titulo)
    ax.set_title(titulo)
    ax.legend()
    ax.set_ylim(0, 1.08)
    ax.axhline(0.90, color='red', ls='--', alpha=0.4, lw=1, label='90%')
    ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparacion_modelos.png', bbox_inches='tight', dpi=150)
plt.show()

print("""
📊 RESPUESTAS A LAS PREGUNTAS DEL WORKSHOP:

¿Cuál modelo tuvo mejor desempeño?
  → El indicado arriba como 🏆. En general, Random Forest y Gradient Boosting
    destacan en datasets tabulares con features bien diseñadas.

¿Alguno presentó overfitting?
  → El Decision Tree sin restricciones tiende a overfitting (memoriza el train).
  → La DNN con EarlyStopping y Dropout debería estar bien regularizada.
  → Se detecta overfitting cuando Train >> Val (gap > 10% en Accuracy o F1).

¿Cuál seleccionaría para producción?
  → El modelo con mejor F1 en VALIDACIÓN y TEST simultáneamente, priorizando
    el RECALL (detectar todos los casos de fatiga es crítico: un falso negativo
    podría resultar en una lesión muscular del ciclista).
""")

---
## Sección 10 – Evaluación Final del Mejor Modelo

### ¿Por qué reentrenamos con Train + Val?
Durante la comparación de modelos, usamos la validación para ajustar hiperparámetros y seleccionar el mejor modelo. Una vez tomada esa decisión, podemos combinar Train y Val para darle al modelo final **más datos de entrenamiento**, lo que generalmente mejora su rendimiento.

El **Test set** se usa ÚNICAMENTE aquí, como evaluación final e imparcial. Este proceso simula cómo funcionaría el modelo con datos completamente nuevos en el mundo real.

In [ ]:
# ── Reentrenamiento del mejor modelo con Train + Val ─────────────────────────
# Combinamos train y val para el entrenamiento final
X_trainval_sc = np.vstack([X_train_sc, X_val_sc])
y_trainval    = pd.concat([y_train, y_val]).reset_index(drop=True)

# Seleccionamos el mejor modelo sklearn (excluimos DNN por simplicidad de reentrenamiento)
resultados_sklearn = {k: v for k, v in zip(
    df_resultados.index,
    df_resultados['F1_Val']
) if k != 'DNN'}
nombre_mejor_sklearn = max(resultados_sklearn, key=resultados_sklearn.get)

print(f'🏆 Mejor modelo sklearn: {nombre_mejor_sklearn}')
print(f'   F1 en validación: {resultados_sklearn[nombre_mejor_sklearn]*100:.2f}%')
print()
print('🔄 Reentrenando con Train + Val...')

# clone() crea una copia del modelo con los mismos hiperparámetros pero sin entrenar
modelo_final = clone(modelos_entrenados[nombre_mejor_sklearn])
modelo_final.fit(X_trainval_sc, y_trainval)

y_pred_final = modelo_final.predict(X_test_sc)

print()
print('=' * 60)
print(f'REPORTE FINAL EN X_TEST – {nombre_mejor_sklearn}')
print('=' * 60)
print(classification_report(
    y_test, y_pred_final,
    target_names=['Normal (0)', 'Fatiga (1)']
))

acc_f  = accuracy_score(y_test, y_pred_final)
prec_f = precision_score(y_test, y_pred_final, zero_division=0)
rec_f  = recall_score(y_test, y_pred_final, zero_division=0)
f1_f   = f1_score(y_test, y_pred_final, zero_division=0)

print(f'  Accuracy  : {acc_f*100:.2f}%')
print(f'  Precision : {prec_f*100:.2f}%')
print(f'  Recall    : {rec_f*100:.2f}%')
print(f'  F1-Score  : {f1_f*100:.2f}%')

In [ ]:
# ── Matriz de Confusión y Boxplots ───────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_final)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'Evaluación Final – {nombre_mejor_sklearn} (reentrenado con Train + Val)',
    fontsize=12, fontweight='bold'
)

# Matriz de confusión
ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Normal (0)', 'Fatiga (1)']
).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(
    'Matriz de Confusión\n'
    f'TN={tn} | FP={fp} | FN={fn} | TP={tp}'
)

# Boxplots de una feature representativa por clase predicha
rms_primera = [c for c in feature_names if c.endswith('_RMS')]
if rms_primera:
    feat_box = rms_primera[0]
    df_test_eval = X_test.copy()
    df_test_eval['pred']  = y_pred_final
    df_test_eval['real']  = y_test.values

    datos_bp = [
        df_test_eval[df_test_eval['pred'] == cls][feat_box].values
        for cls in [0, 1]
    ]
    bp = axes[1].boxplot(datos_bp, patch_artist=True, notch=False, widths=0.5)
    for caja, color in zip(bp['boxes'], ['#1565C0', '#C62828']):
        caja.set_facecolor(color)
        caja.set_alpha(0.65)
    axes[1].set_xticklabels(['Predicho: Normal', 'Predicho: Fatiga'])
    axes[1].set_title(
        f'Distribución de {feat_box}\n'
        f'según clase predicha (confirma separabilidad)'
    )
    axes[1].set_ylabel('Valor')

plt.tight_layout()
plt.savefig('evaluacion_final.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"""
📊 ANÁLISIS DE LA MATRIZ DE CONFUSIÓN:

  Verdaderos Negativos (TN) = {tn:>5}  → Normal clasificado correctamente como Normal
  Falsos Positivos     (FP) = {fp:>5}  → Normal clasificado incorrectamente como Fatiga
                                          (alarma falsa: puede generar paradas innecesarias)
  Falsos Negativos     (FN) = {fn:>5}  → Fatiga clasificada incorrectamente como Normal ⚠️
                                          (CRÍTICO: el ciclista continúa fatigado sin saberlo)
  Verdaderos Positivos (TP) = {tp:>5}  → Fatiga clasificada correctamente como Fatiga

El error más costoso en este contexto es el Falso Negativo:
si no detectamos fatiga cuando existe, el ciclista podría sufrir una lesión muscular.
Por eso maximizamos el RECALL (= TP / (TP + FN)).

¿Es un buen clasificador?
  → Si F1 > 85% y Recall > 80%: SÍ, es útil para detección de fatiga en tiempo real.

¿Cómo podría mejorarse?
  → Agregar features wavelet (mejor resolución tiempo-frecuencia)
  → Usar ventanas solapadas (más datos de entrenamiento)
  → Aplicar SMOTE si el dataset está desbalanceado
  → Usar LSTM/GRU directamente sobre la señal cruda (captura dependencias temporales)
  → Validación Leave-One-Subject-Out para evaluar generalización a nuevos sujetos
""")

---
## Sección 11 – Prueba con Muestra Artificial

Para verificar que el modelo aprendió patrones reales (y no aleatoriedad del conjunto de entrenamiento), generamos muestras sintéticas con valores fisiológicamente plausibles y las clasificamos.

### ¿Cómo generamos las muestras?
Tomamos percentiles del dataset real para construir perfiles representativos:
- **Perfil de fatiga**: valores del **percentil 75** de la clase fatiga (claramente fatigado).
- **Perfil normal**: valores del **percentil 25** de la clase normal (claramente saludable).

Al usar percentiles en lugar de valores aleatorios, garantizamos que las muestras sean fisiológicamente realistas.

In [ ]:
print('=== PRUEBA CON MUESTRAS ARTIFICIALES ===')
print()

# Separamos las features reales por clase
feats_normales = df_features[df_features['target'] == 0][feature_names]
feats_fatiga   = df_features[df_features['target'] == 1][feature_names]

# Construimos el perfil de fatiga con el percentil 75 de la clase fatiga
# → valores que el 75% de las muestras de fatiga tienen por debajo
# → representa un caso claramente fatigado
muestra_fatiga = feats_fatiga.quantile(0.75).to_frame().T

# Construimos el perfil normal con el percentil 25 de la clase normal
# → valores que el 25% de las muestras normales tienen por debajo
# → representa un caso claramente saludable
muestra_normal = feats_normales.quantile(0.25).to_frame().T

# Escalamos con el mismo preprocesador entrenado (IMPORTANTE: no re-ajustar)
muestra_fatiga_sc = preprocessor.transform(muestra_fatiga)
muestra_normal_sc = preprocessor.transform(muestra_normal)

# Predicciones
pred_fatiga = modelo_final.predict(muestra_fatiga_sc)[0]
pred_normal = modelo_final.predict(muestra_normal_sc)[0]

# Probabilidades (si el modelo las soporta)
prob_fatiga = None
prob_normal = None
if hasattr(modelo_final, 'predict_proba'):
    prob_fatiga = modelo_final.predict_proba(muestra_fatiga_sc)[0][1]  # prob de clase 1
    prob_normal = modelo_final.predict_proba(muestra_normal_sc)[0][1]

# Mostrar resultados
print('─' * 50)
print('MUESTRA ARTIFICIAL – PERFIL DE FATIGA')
print('(Construida con el percentil 75 de la clase fatiga)')
print(f'  Predicción: {"⚠️  FATIGA MUSCULAR" if pred_fatiga == 1 else "✅ CONDICIÓN NORMAL"}')
if prob_fatiga is not None:
    print(f'  Probabilidad de fatiga: {prob_fatiga*100:.1f}%')
    print(f'  ¿Resultado esperado?: {"✅ SÍ (predijo fatiga correctamente)" if pred_fatiga == 1 else "❌ NO (debería haber predicho fatiga)"}')

print()
print('─' * 50)
print('MUESTRA ARTIFICIAL – PERFIL NORMAL')
print('(Construida con el percentil 25 de la clase normal)')
print(f'  Predicción: {"⚠️  FATIGA MUSCULAR" if pred_normal == 1 else "✅ CONDICIÓN NORMAL"}')
if prob_normal is not None:
    print(f'  Probabilidad de fatiga: {prob_normal*100:.1f}%')
    print(f'  ¿Resultado esperado?: {"✅ SÍ (predijo normal correctamente)" if pred_normal == 0 else "❌ NO (debería haber predicho normal)"}')

print("""
📊 ANÁLISIS:

Las muestras artificiales se construyeron con percentiles reales del dataset, por lo que
representan perfiles fisiológicamente plausibles. Hay dos posibles escenarios:

✅ Si ambas muestras se clasifican correctamente:
   El modelo ha aprendido los patrones reales de la señal EMG y no está memorizando
   el entrenamiento. En producción, recibiría señales de sensores en tiempo real.

⚠️  Si alguna muestra se clasifica incorrectamente:
   Puede indicar que el modelo necesita más regularización, más datos, o que las
   features seleccionadas no capturan suficientemente bien el fenómeno de fatiga.

En un sistema real de ciclismo:
   Los sensores EMG enviarían datos continuamente → cada segundo se extraerían
   las 8 features por canal → el modelo clasificaría en tiempo real → si detecta
   fatiga, alertaría al ciclista o al entrenador.
""")

---
## Sección 12 – Conclusiones

### Resumen del proyecto

Desarrollamos un sistema completo de detección de fatiga muscular en ciclistas a partir de señales EMG de 8 canales. El flujo fue:

**1. Feature Engineering:** Transformamos series de tiempo de alta frecuencia en un dataset tabular usando ventanas de 1 segundo y 8 características por canal (RMS, VAR, ZCR, MAV, WAMP, MNF, MDF, TTP), combinando el dominio del tiempo y la frecuencia.

**2. EDA:** Confirmamos visualmente que las características de energía (RMS) y frecuencia (MNF, MDF) son los mejores discriminadores entre clases, alineado con la literatura de fisiología del ejercicio.

**3. Modelos:** Entrenamos 5 clasificadores con ajuste de hiperparámetros via Randomized Search. Los modelos basados en árboles (Random Forest, Gradient Boosting) generalmente superan a kNN y al árbol individual en datos tabulares.

**4. Evaluación:** El modelo final se evalúa con métricas completas (Accuracy, Precision, Recall, F1) con énfasis en el Recall por la naturaleza crítica del problema.

### Limitaciones y mejoras posibles

- **Ventanas solapadas**: usar un paso < window_size generaría más muestras de entrenamiento.
- **Features wavelet**: la transformada wavelet ofrece mejor resolución tiempo-frecuencia que la FFT.
- **Modelos secuenciales**: LSTM o GRU podrían capturar dependencias temporales entre ventanas.
- **Validación por sujeto**: Leave-One-Subject-Out CV evalúa la generalización a nuevos individuos.
- **SMOTE**: si el dataset está desbalanceado, generar muestras sintéticas de la clase minoritaria.
- **Ensemble de modelos**: combinar las predicciones de múltiples modelos (stacking) puede mejorar resultados.